# Week 7: reconstruction and generation with a VAE

Run `python vae.py` from the Module 3 folder first. This notebook uses the synthetic Module 2 attachment images; they are not customer data.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt
from PIL import Image
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from vae import VAE, AttachmentDataset, DEFAULT_MANIFEST
checkpoint = torch.load(ROOT / "artifacts/vae/model.pt", map_location="cpu", weights_only=True)
model = VAE(checkpoint["latent_dim"])
model.load_state_dict(checkpoint["state_dict"])
model.eval()


## Reconstruction
The encoder maps an existing image to a latent distribution. Here we decode its mean. The upper row is the original; the lower row is the reconstruction.


In [ ]:
dataset = AttachmentDataset(DEFAULT_MANIFEST, "validation")
original = torch.stack([dataset[i] for i in range(8)])
with torch.no_grad():
    mu, logvar = model.encode(original)
    reconstructed = model.decoder(mu)
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    for row, batch in enumerate((original, reconstructed)):
        axes[row, i].imshow(batch[i].permute(1, 2, 0).numpy())
        axes[row, i].axis("off")
axes[0, 0].set_title("Original")
axes[1, 0].set_title("Reconstruction")
plt.tight_layout()


## Sampling
Sampling draws a new latent vector from the prior, without supplying an input image. These outputs are new synthetic images, although they can resemble training examples.


In [ ]:
torch.manual_seed(43)
with torch.no_grad():
    samples = model.sample(8, torch.device("cpu"))
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(samples[i].permute(1, 2, 0).numpy())
    ax.axis("off")
plt.tight_layout()


## Latent interpolation
The decoder turns intermediate points between two encoded images into a gradual visual transition. This illustrates a continuous latent space, not a semantic guarantee.


In [ ]:
with torch.no_grad():
    z0, _ = model.encode(original[:1])
    z1, _ = model.encode(original[1:2])
    alpha = torch.linspace(0, 1, 8).unsqueeze(1)
    interpolated = model.decoder((1-alpha)*z0 + alpha*z1)
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(interpolated[i].permute(1, 2, 0).numpy())
    ax.axis("off")
plt.tight_layout()


## Questions
1. Which small text and shapes disappear in reconstructions?
2. How do samples differ from reconstructions?
3. What changes if `--beta` increases? Compare reconstruction fidelity and sample structure after another run.
